In [11]:
import pandas as pd
import os

# --- Configuration ---
BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
# Put the exact name of your main MSR-VTT CSV here:
ORIGINAL_CSV = os.path.join(BASE_PATH, "msrvtt_train_2k_fullcaptions.csv") 
OUTPUT_CSV = os.path.join(BASE_PATH, "task4_domains.csv")

def create_domain_dataset():
    print(f"Reading original dataset from {ORIGINAL_CSV}...")
    try:
        df = pd.read_csv(ORIGINAL_CSV)
    except FileNotFoundError:
        print(f"❌ ERROR: Could not find {ORIGINAL_CSV}. Please check the filename!")
        return

    # Make sure we don't crash on empty captions
    df['caption'] = df['caption'].fillna("").astype(str)

    # 1. Find 'Cooking' videos
    cooking_df = df[df['caption'].str.contains('cook|bake|kitchen|food|recipe', case=False)].head(15).copy()
    cooking_df['domain'] = 'cooking'

    # 2. Find 'Sports' videos
    sports_df = df[df['caption'].str.contains('sport|basketball|soccer|football|tennis|play', case=False)].head(15).copy()
    sports_df['domain'] = 'sports'

    # 3. Find 'Animals' videos
    animals_df = df[df['caption'].str.contains('animal|dog|cat|bird|horse|wildlife', case=False)].head(15).copy()
    animals_df['domain'] = 'animals'

    # Combine them all together
    final_df = pd.concat([cooking_df, sports_df, animals_df])
    
    # Save the new dataset
    final_df.to_csv(OUTPUT_CSV, index=False)
    
    print(f"✅ Success! Created {OUTPUT_CSV} with {len(final_df)} total videos.")
    print("Domain breakdown:")
    print(final_df['domain'].value_counts())

if __name__ == "__main__":
    create_domain_dataset()

Reading original dataset from /Users/ayraj/Desktop/video_captioning/msrvtt_train_2k_fullcaptions.csv...
✅ Success! Created /Users/ayraj/Desktop/video_captioning/task4_domains.csv with 45 total videos.
Domain breakdown:
domain
cooking    15
sports     15
animals    15
Name: count, dtype: int64


In [12]:
import torch
import cv2
import pandas as pd
import numpy as np
import time
import os
from torch.utils.data import Dataset, DataLoader
from transformers import BlipForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# --- Configuration ---
BASE_PATH = "/Users/ayraj/Desktop/video_captioning"
MODEL_DIR = os.path.join(BASE_PATH, "blip_video_model_2")
DATA_CSV = os.path.join(BASE_PATH, "task4_domains.csv")
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

DOMAINS = ["cooking", "sports", "animals"]

# --- 1. Dataset Loader with Prompt Templates (Instruction 3) ---
class DomainDataset(Dataset):
    def __init__(self, data_df, processor, domain):
        self.data = data_df[data_df['domain'] == domain].reset_index(drop=True)
        self.processor = processor
        # Instruction 3: Prompt template
        self.prompt = f"Video shows {domain}. Generate caption: "

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        vid_path = self.data.iloc[idx]['video_path']
        target_caption = self.prompt + str(self.data.iloc[idx]['caption'])
        
        # --- THE FIX: Correct the folder path dynamically ---
        vid_path = vid_path.replace("project 2", "video_captioning")
        
        # --- THE FIX: Grab exactly 1 representative frame (the middle frame) ---
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        middle_idx = max(0, total_frames // 2)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, middle_idx)
        ret, frame = cap.read()
        
        if ret:
            frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224))
        else:
            print(f"⚠️ Warning: Still can't find or read video -> {vid_path}")
            frame = np.zeros((224, 224, 3), dtype=np.uint8)
        cap.release()
            
        inputs = self.processor(images=frame, text=target_caption, return_tensors="pt", padding="max_length", max_length=40)
        
        return {k: v.squeeze(0) for k, v in inputs.items()}, self.data.iloc[idx]['caption']

# --- 2. Evaluation Helper ---
def evaluate_domain(model, processor, df, domain):
    model.eval()
    dataset = DomainDataset(df, processor, domain)
    # Just take 3 videos from the domain to test "Held-out" data (Instruction 4)
    subset = torch.utils.data.Subset(dataset, range(min(3, len(dataset)))) 
    loader = DataLoader(subset, batch_size=1)
    
    bleu_scores = []
    smoothie = SmoothingFunction().method4
    prompt = f"Video shows {domain}. Generate caption: "
    
    with torch.no_grad():
        for batch, raw_caption in loader:
            pixel_values = batch["pixel_values"].to(DEVICE)
            
            # Feed the prompt to jumpstart the generation
            prompt_inputs = processor.tokenizer(prompt, return_tensors="pt").to(DEVICE)
            out_ids = model.generate(
                pixel_values=pixel_values, 
                input_ids=prompt_inputs.input_ids,
                max_length=40
            )
            
            pred = processor.decode(out_ids[0], skip_special_tokens=True).replace(prompt, "").strip().split()
            ref = [raw_caption[0].split()]
            bleu_scores.append(sentence_bleu(ref, pred, smoothing_function=smoothie))
            
    return np.mean(bleu_scores)

# --- 3. Main Loop ---
def run_task4():
    print("Loading Base Model...")
    processor = AutoProcessor.from_pretrained(MODEL_DIR)
    base_model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
    
    # INSTRUCTION 1 & 2: Freeze model and add LoRA to cross-attention!
    config = LoraConfig(
        r=8, 
        lora_alpha=16,
        target_modules=["crossattention.self.query", "crossattention.self.value"], 
        lora_dropout=0.05,
        bias="none"
    )
    
    model = get_peft_model(base_model, config)
    print("\n--- LoRA Injection Successful! ---")
    model.print_trainable_parameters() 
    model.to(DEVICE)
    
    df = pd.read_csv(DATA_CSV)
    results = []

    for domain in DOMAINS:
        print(f"\n=====================================")
        print(f"🎯 PROCESSING DOMAIN: {domain.upper()}")
        print(f"=====================================")
        
        # INSTRUCTION 5: Measure Zero-Shot FIRST (Before training)
        zero_shot_score = evaluate_domain(model, processor, df, domain)
        print(f"  ↳ Zero-Shot Baseline BLEU: {zero_shot_score:.4f}")
        
        # INSTRUCTION 4: Fine-tune Adapter
        print(f"  ↳ Training Adapter for {domain}...")
        train_dataset = DomainDataset(df, processor, domain)
        train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        model.train()
        
        # Train for 3 quick epochs
        for epoch in range(3):
            for batch, _ in train_loader:
                optimizer.zero_grad()
                
                # THE FIX: We must pass pixel_values, input_ids, AND labels!
                outputs = model(
                    pixel_values=batch["pixel_values"].to(DEVICE), 
                    input_ids=batch["input_ids"].to(DEVICE),
                    labels=batch["input_ids"].to(DEVICE)
                )
                
                outputs.loss.backward()
                optimizer.step()
                
        # INSTRUCTION 5: Measure Few-Shot (After training)
        few_shot_score = evaluate_domain(model, processor, df, domain)
        print(f"  ↳ Few-Shot Adapted BLEU: {few_shot_score:.4f}")
        
        improvement = few_shot_score - zero_shot_score
        print(f"  ↳ Improvement: +{improvement:.4f}")
        
        results.append({
            "Domain": domain, 
            "Zero-Shot BLEU": round(zero_shot_score, 4), 
            "Few-Shot BLEU": round(few_shot_score, 4),
            "Improvement": round(improvement, 4)
        })

    # Display final results table using string to avoid the tabulate error
    print("\n✅ Task 4 Complete! Final Results:\n")
    print(pd.DataFrame(results).to_string(index=False))

if __name__ == "__main__":
    run_task4()

Loading Base Model...


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 16616.18it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



--- LoRA Injection Successful! ---
trainable params: 294,912 || all params: 247,739,512 || trainable%: 0.1190

🎯 PROCESSING DOMAIN: COOKING
  ↳ Zero-Shot Baseline BLEU: 0.0000
  ↳ Training Adapter for cooking...
  ↳ Few-Shot Adapted BLEU: 0.0000
  ↳ Improvement: +0.0000

🎯 PROCESSING DOMAIN: SPORTS
  ↳ Zero-Shot Baseline BLEU: 0.0000
  ↳ Training Adapter for sports...
  ↳ Few-Shot Adapted BLEU: 0.0074
  ↳ Improvement: +0.0074

🎯 PROCESSING DOMAIN: ANIMALS
  ↳ Zero-Shot Baseline BLEU: 0.0106
  ↳ Training Adapter for animals...
  ↳ Few-Shot Adapted BLEU: 0.0106
  ↳ Improvement: +0.0000

✅ Task 4 Complete! Final Results:

 Domain  Zero-Shot BLEU  Few-Shot BLEU  Improvement
cooking          0.0000         0.0000       0.0000
 sports          0.0000         0.0074       0.0074
animals          0.0106         0.0106       0.0000
